1 ToolStrategy的多种结构化输出方式Schema参数

1.1 Pydantic类型

In [7]:
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class ContractInfo(BaseModel):
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(schema=ContractInfo),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='217c2659-b453-4657-898a-59f7d2956b46'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 384,
                    'total_tokens': 450,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 384
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-222c7d76-1b4a-931c-b73b-f5bafbb8ee9d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa4e-cb10-7a70-9208-d7fdecd5c5e1-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_11ad554198c4447fa8d4a3c5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 384,
                'output_tokens': 66,
                'total_tokens': 450,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='1414227322@qq.com' phone='18788483333'",
            name='ContractInfo',
            id='03b68dab-ec56-4874-9160-a29c9d13c212',
            tool_call_id='call_11ad554198c4447fa8d4a3c5'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='1414227322@qq.com', phone='18788483333')
}

1.2 TypeDict类型

In [10]:
from typing import Literal,TypedDict,Annotated
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
class ContractInfo(TypedDict):
    """
    提取用户的联系方式
    """
    name: Annotated[str,...,"用户姓名"]
    email:Annotated[str,...,"用户邮箱号"]
    phone:Annotated[str,...,"用户手机"]
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(schema=ContractInfo),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='1a2d9947-8a09-4fd4-bf36-74e735f97af0'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 362,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 362
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-355cc0f5-6c6f-9c45-8d07-bd86335459d5',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa55-d307-7780-a69c-5d3524154996-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_f49480e62f2d4974ab6bab9a',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 362,
                'output_tokens': 66,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': '1414227322@qq.com', 'phone': 
'18788483333'}",
            name='ContractInfo',
            id='abecb9cf-09a0-48d5-848d-7a62eb58f233',
            tool_call_id='call_f49480e62f2d4974ab6bab9a'
        )
    ],
    'structured_response': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'}
}

1.3 JsonSchema

In [11]:
from typing import Literal,TypedDict,Annotated
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

json_schema={
    'title':"ContractInfo",
    'type': 'object',
    'description':"用户的联系方式" ,
    'properties': {
        'name': {
            'description': "用户名称",
            'type': 'string'
        },
        'email': {
            'description': '用户邮箱号',
            'type': 'string'
        },
        'phone': {
            'description': '用户手机号',
            'type': 'string'
        }
    },
    'required': ['name','email','phone']
}
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(schema=json_schema),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='b5ac44e3-5a77-4cc4-b89f-5cfe3c5e9c4a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 381,
                    'total_tokens': 447,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 381
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-0139d5b3-e5fa-9adc-9d4f-e1cb02d24ae6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa5b-6fe2-7c11-8193-a6657a471aaa-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_bfdc042b24574c6dbd1d4dcc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 381,
                'output_tokens': 66,
                'total_tokens': 447,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': '1414227322@qq.com', 'phone': 
'18788483333'}",
            name='ContractInfo',
            id='c3ac4068-b0ef-4f93-8051-51b76a8213f5',
            tool_call_id='call_bfdc042b24574c6dbd1d4dcc'
        )
    ],
    'structured_response': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'}
}

1.4@dataclass类型

In [15]:
from typing import Literal,TypedDict,Annotated
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from dataclasses import dataclass
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")

#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(schema=json_schema),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            [                HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333',
            additional_kwargs={},
            response_metadata={},
            id='ba106a42-b831-472e-a24d-caa6d6381a18'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 66,
                    'prompt_tokens': 381,
                    'total_tokens': 447,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 381
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-1007a3d6-faae-962e-a3ed-002d5f775947',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa5e-4af6-7191-baf5-56ab143b9dc3-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'},
                    'id': 'call_986e7bb138f44e569a3d5609',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 381,
                'output_tokens': 66,
                'total_tokens': 447,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': '1414227322@qq.com', 'phone': 
'18788483333'}",
            name='ContractInfo',
            id='0be03704-f160-46e8-91c7-971f1873ebc9',
            tool_call_id='call_986e7bb138f44e569a3d5609'
        )
    ],
    'structured_response': {'name': '小明', 'email': '1414227322@qq.com', 'phone': '18788483333'}
}

1.5 联合模式 Union[type1,type2]

In [17]:
from typing import Literal,TypedDict,Annotated,Union
from langchain.agents.structured_output import ProviderStrategy,ToolStrategy,AutoStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from dataclasses import dataclass
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
    reasoning_effort="none"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@dataclass
class ContractInfo:
    """
    提取用户的联系方式
    """
    name:str=Field(description="用户姓名")
    email:str=Field(description="用户邮箱号")
    phone:str=Field(description="用户手机号")
@dataclass
class EventInfo:
    """
    事件详情
    """
    event_name:str=Field(description="事件名称")
    date:str=Field(description="事件发送日期")


#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    name="agent01",
    tools=[],
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的信息处理小助手,当有人问你关于信息处理问题的时候，你会帮助询问者完成信息处理任务。"),
    response_format=ToolStrategy(schema=Union[ContractInfo,EventInfo]),
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错
        "messages":
            # [HumanMessage("请你提取一下该员工的联系信息：小明的邮箱是1414227322@qq.com,手机是18788483333")]
            [HumanMessage("请你提取该句子中的结构信息：2026年7月9日高考人数已经突破1200万")]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请你提取该句子中的结构信息：2026年7月9日高考人数已经突破1200万',
            additional_kwargs={},
            response_metadata={},
            id='6fe32a1e-1a14-4d43-bc26-47316c37cd66'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 50,
                    'prompt_tokens': 456,
                    'total_tokens': 506,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 456
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-5054c9b1-fc98-9813-8eb8-73b58f90495e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0aa64-74bf-7842-a128-7e5da58c22f1-0',
            tool_calls=[
                {
                    'name': 'EventInfo',
                    'args': {'event_name': '高考人数已经突破1200万', 'date': '2026年7月9日'},
                    'id': 'call_bf32ed86bbea48948897b76a',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 456,
                'output_tokens': 50,
                'total_tokens': 506,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: EventInfo(event_name='高考人数已经突破1200万', 
date='2026年7月9日')",
            name='EventInfo',
            id='cb465c65-4d8f-4c4b-81f9-a7e1962252a8',
            tool_call_id='call_bf32ed86bbea48948897b76a'
        )
    ],
    'structured_response': EventInfo(event_name='高考人数已经突破1200万', date='2026年7月9日')
}